# DR grading: exact paper protocol on Colab (300 px, 60 epochs)

Reproduces Chilukoti et al. (2024), strategy 2: EfficientNet-B3, CLAHE + Gaussian blur, **300 x 300**, **60 epochs**, batch 16, Adam lr 1e-3, weight decay 1e-4, grad-clip 0.1, 10-snapshot majority vote. A second run repeats it with lr 1e-4.

**Before starting:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**.

Everything important (data, checkpoints, results) is saved to Google Drive in `MyDrive/DR_colab/`. If Colab disconnects, reconnect and **Run all** again: finished runs are skipped and an unfinished run resumes from its last epoch.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
# 2. Mount Google Drive (approve the pop-up)
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/DR_colab'
!mkdir -p {DRIVE}/data {DRIVE}/runs {DRIVE}/logs

In [ ]:
# 3. Get the code from GitHub and install the two missing packages
import os
if not os.path.exists('/content/dr-grading-reproduction'):
    !git clone -q https://github.com/sshivanshg/dr-grading-reproduction.git /content/dr-grading-reproduction
else:
    !git -C /content/dr-grading-reproduction pull -q
%cd /content/dr-grading-reproduction/dr-repro
!pip -q install timm opencv-python-headless

In [ ]:
# 4. Keep runs/, logs/ and results/ on Drive so nothing is lost on disconnect
import os, shutil
!mkdir -p {DRIVE}/results
for d in ['runs', 'logs', 'results']:
    if os.path.islink(d):
        continue
    if os.path.isdir(d):
        for item in os.listdir(d):
            dst = f'{DRIVE}/{d}/{item}'
            if not os.path.exists(dst):
                (shutil.copytree if os.path.isdir(f'{d}/{item}') else shutil.copy)(f'{d}/{item}', dst)
        shutil.rmtree(d)
    os.symlink(f'{DRIVE}/{d}', d)
!ls -la runs logs results

In [ ]:
# 5. Data: restore from Drive if saved before, otherwise download APTOS (~9 GB, ~10-20 min) and save a copy
import os
if os.path.exists(f'{DRIVE}/data/aptos512.tar'):
    !mkdir -p data && tar -xf {DRIVE}/data/aptos512.tar -C data && cp {DRIVE}/data/labels.csv data/
else:
    !python -u scripts/prepare_data.py
    !tar -cf {DRIVE}/data/aptos512.tar -C data aptos512 && cp data/labels.csv {DRIVE}/data/
!ls data/aptos512 | wc -l && wc -l data/labels.csv

In [ ]:
# 6. TRAIN. Two runs of 60 epochs each; per-epoch lines are printed as JSON.
#    If this stops (disconnect / timeout), just Run all again: it resumes.
!bash scripts/run_colab.sh

In [ ]:
# 7. Summary of test results
import json, os
rows = [('s2_paper300', 'paper settings, lr 1e-3'), ('s2_paper300_lr1e4', 'lr 1e-4')]
print(f"{'run':28s}{'prediction':16s}{'QWK':>7s}{'acc':>7s}{'mF1':>7s}{'AUC':>7s}{'Sev':>6s}{'PDR':>6s}")
for run, label in rows:
    p = f'runs/{run}/results.json'
    if not os.path.exists(p):
        print(run, 'not finished'); continue
    r = json.load(open(p))
    for k in ['final_epoch', 'best_val', 'ensemble_vote', 'ensemble_mean']:
        m = r[k]; pc = m['per_class']
        print(f"{label:28s}{k:16s}{m['qwk']:7.3f}{m['accuracy']:7.3f}{m['macro_f1']:7.3f}{m['macro_auc_ovr']:7.3f}"
              f"{pc['Severe']['recall']:6.2f}{pc['PDR']['recall']:6.2f}")
print('paper (reported): QWK 0.954 single model, 0.967 with EyePACS + ensemble')

In [ ]:
# 8. Package results (also saved to Drive as DR_colab/results_colab.zip)
!zip -q -r {DRIVE}/results_colab.zip runs/s2_paper300* logs/s2_paper300* results/figures results/leakage_s2_paper300*.json -x '*.pt'
from google.colab import files
files.download(f'{DRIVE}/results_colab.zip')